# Diagnose & Fix Sanskrit RAG Errors

This notebook walks through reproducing the failure, analyzing tracebacks, inspecting the environment, updating requirements, modifying code, and finally verifying the fix. It mirrors the troubleshooting steps needed for the `simple_rag_demo.py` / `ollama_rag_ui.py` errors.


## 1. Setup and Reproduce Error

Install dependencies if not yet done, then run the failing script to trigger the error. We'll reproduce the `datasets.download` crash as shown earlier.


In [ ]:
```python
# make sure requirements are installed
!pip install -r "Data Scrapping/requirements.txt"

# run the CLI script with a test query (this will show the traceback)
!python "Data Scrapping/simple_rag_demo.py" "test"
```

## 2. Analyze Error Traceback

The traceback pinpoints `ModuleNotFoundError: No module named 'datasets.download'` during `import sentence_transformers`. We'll capture the full error text from the previous run.


The important lines:

```
ModuleNotFoundError: No module named 'datasets.download'
  File ".../sentence_transformers/model_card.py", line 33, in <module>
    from datasets import Dataset, DatasetDict, Value
  File ".../datasets/arrow_reader.py", line 30, in <module>
    from .download.download_config import DownloadConfig
```

This indicates the `datasets` package is missing the `download` submodule.


## 3. Inspect Dependencies and Environment

Run `pip list` and show the `requirements.txt` contents; verify Python version.
Also include instructions for creating a virtual environment for isolation.


In [ ]:
```bash
# list installed packages
pip list

# show requirements
sed -n '1,30p' "Data Scrapping/requirements.txt"

# python version
python --version

# create a virtual env example
python -m venv venv
.\venv\Scripts\activate
pip install -r "Data Scrapping/requirements.txt"
```

## 4. Update Requirements and Fix Imports

The error stems from a broken `datasets` install. We can either pin a working version or patch the import.

```bash
# try force-reinstall a known-good version
pip install "datasets==2.12.0" --force-reinstall

# alternative: modify simple_rag_demo.py to bypass sentence-transformers import
# by lazy loading or catching import errors
```


## 5. Modify Code to Handle Missing Files/Paths

Add guard clauses in `simple_rag_demo.py`:

```python
if not os.path.isdir(nalanda_path):
    os.makedirs(nalanda_path, exist_ok=True)
    logger.warning("Corpus directory was missing; created empty folder.")

# in loader: raise FileNotFoundError if input path is invalid
```


## 6. Run Sample Queries to Verify Fix

After applying changes and installing a proper `datasets` version, run the CLI or UI again. Example test:

```python
!python "Data Scrapping/simple_rag_demo.py" "किम्"  # Sanskrit test
```

Add assertions if writing unit tests, e.g.:

```python
assert "answer" in output.lower()
```

```

```
